# Tests de recherche — vérification de l'efficacité de l'index FAISS

Charge l'index construit par `build_index.py` et interroge-le avec des questions représentatives d'un utilisateur du chatbot, pour vérifier concrètement que la recherche par similarité sémantique retrouve des événements pertinents.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings

load_dotenv(Path("..") / ".env")
api_key = os.getenv("MISTRAL_API_KEY")
assert api_key, "MISTRAL_API_KEY manquant dans .env"

embeddings = MistralAIEmbeddings(mistral_api_key=api_key)

INDEX_PATH = Path("..") / "data" / "index"
# allow_dangerous_deserialization=True : sûr ici, l'index vient de build_index.py
vector_store = FAISS.load_local(str(INDEX_PATH), embeddings, allow_dangerous_deserialization=True)

print("Index chargé, nombre de vecteurs :", vector_store.index.ntotal)

C:\Users\faiza\AppData\Local\Temp\ipykernel_32924\176330960.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Index chargé, nombre de vecteurs : 7169


## Fonction d'affichage des résultats

`similarity_search_with_score` renvoie une liste de `(Document, score)` — le score est une **distance L2** (plus il est petit, plus le chunk est proche sémantiquement de la question).

In [2]:
def run_query(question: str, k: int = 3) -> None:
    print(f"### Question : {question!r}\n")
    results = vector_store.similarity_search_with_score(question, k=k)
    for rank, (doc, score) in enumerate(results, start=1):
        meta = doc.metadata
        print(f"{rank}. {meta.get('title')}  (score={score:.3f})")
        print(f"   date  : {meta.get('date_start')}")
        print(f"   lieu  : {meta.get('location_city')}")
        print(f"   statut: {meta.get('status')}  | conditions: {meta.get('conditions')}")
        print(f"   extrait: {doc.page_content[:150]!r}")
        print()

## Question 1 — recherche thématique + lieu

In [3]:
run_query("concert de musique à Marseille")

### Question : 'concert de musique à Marseille'

1. Concert Musicâme à Marseille : Debussy, Bizet, Aznavour, Piaff, Saint-Saëns, Fauré, Ravel  (score=0.308)
   date  : 2026-03-13T19:30:00+00:00
   lieu  : Marseille
   statut: Programmé  | conditions: None
   extrait: 'Mots-clés : musique classique, concert marseille, marseille, concert, musicame france'

2. ! COMPLET ! Babel Music XP - Concerts  (score=0.330)
   date  : 2026-03-19T19:30:00+00:00
   lieu  : Marseille
   statut: Programmé  | conditions: None
   extrait: '! COMPLET ! Babel Music XP - Concerts\n\nÉvénement des bibliothèques de Marseille'

3. Concert à Marseille : Ravel, Debussy, Cantemir, Mozart, Bach, Puccini, Piazzolla, Vivaldi  (score=0.342)
   date  : 2026-07-24T18:30:00+00:00
   lieu  : Marseille
   statut: Programmé  | conditions: De 20 à 30€
   extrait: 'Conditions/tarifs : De 20 à 30€\n\nMots-clés : musicame france, concert musique classique, juillet marseille, marseille'



## Question 2 — filtre implicite sur le prix

In [4]:
run_query("événement gratuit pour les enfants")

### Question : 'événement gratuit pour les enfants'

1. Restitution de la performance "Créatures :  la monstruosité en mouvement, entre photographie et performance"  (score=0.409)
   date  : 2027-05-21T17:00:00+00:00
   lieu  : Marseille
   statut: Programmé  | conditions: Évènement gratuit, sur réservations
   extrait: 'Conditions/tarifs : Évènement gratuit, sur réservations'

2. Jeux pour les tout petits et petites  (score=0.428)
   date  : 2026-05-09T08:00:00+00:00
   lieu  : Marseille
   statut: Programmé  | conditions: None
   extrait: 'Jeux pour les tout petits et petites\n\nÉvénement des bibliothèques de Marseille\n\n<p>Nous invitons les tous petits pour une matinée jeux !<br>Entrée lib'

3. JEUX PEDALE EN FAMILLE  (score=0.447)
   date  : 2026-05-30T08:00:00+00:00
   lieu  : Meyreuil
   statut: Programmé  | conditions: None
   extrait: 'JEUX PEDALE EN FAMILLE\n\nJournée gratuite de découverte avec Kart à pédales pour petits et grands, Circuit vélo enfants, Initiation vélo route

## Question 3 — thème culturel différent (exposition/patrimoine)

In [5]:
run_query("exposition d'art ou visite du patrimoine")

### Question : "exposition d'art ou visite du patrimoine"

1. REGARDS ARTISTIQUES  SUR LE PATRIMOINE ET L'ARCHITECTURE DE MARSEILLE  (score=0.400)
   date  : 2025-09-20T08:00:00+00:00
   lieu  : Marseille
   statut: Programmé  | conditions: Entrée libre
   extrait: '<p>Pour cette nouvelle édition des Journées du Patrimoine, le Centre Cormier et l’Atelier Cézanne vous proposent de redécouvrir le patrimoine et l’arc'

2. Visite libre au Musée Réattu  (score=0.414)
   date  : 2026-09-19T08:00:00+00:00
   lieu  : Arles
   statut: Programmé  | conditions: None
   extrait: "Visite libre au Musée Réattu\n\nÀ l'occasion des Journées européennes du patrimoine 2026, le musée Réattu propose quatre rendez-vous autour du patrimoin"

3. Friche la Belle de Mai : les expositions d'art contemporain  (score=0.418)
   date  : 2026-09-19T12:00:00+00:00
   lieu  : Marseille
   statut: Programmé  | conditions: None
   extrait: "Friche la Belle de Mai : les expositions d'art contemporain\n\nVisitez gratuitem

## Question 4 — vérifie que le statut "Annulé" est bien retrouvable

Teste directement ce que `build_text()` a été conçu pour permettre : répondre à une question sur l'annulation d'un événement.

In [6]:
run_query("cet événement a-t-il été annulé ?")

### Question : 'cet événement a-t-il été annulé ?'

1. Les balades à vélo du MauMa  (score=0.401)
   date  : 2026-05-09T08:00:00+00:00
   lieu  : Marseille
   statut: Programmé  | conditions: None
   extrait: 'concernant la sortie. Pensez à vérifier vos spams.<br>❌ Les annulation de dernière minute 48h avant l’évènement ne pourront être remboursées.</p>'

2. Atelier d'écriture  (score=0.405)
   date  : 2026-04-08T12:30:00+00:00
   lieu  : Marseille
   statut: Annulé  | conditions: None
   extrait: "Statut de l'événement : Annulé.\n\nAtelier d'écriture\n\nÉvénement des bibliothèques de Marseille\n\n<p>Un rendez-vous mensuel : atelier d'écriture pour déb"

3. Atelier d'écriture  (score=0.405)
   date  : 2026-03-28T09:30:00+00:00
   lieu  : Marseille
   statut: Annulé  | conditions: None
   extrait: "Statut de l'événement : Annulé.\n\nAtelier d'écriture\n\nÉvénement des bibliothèques de Marseille\n\n<p>Un rendez-vous mensuel : atelier d'écriture pour déb"



## Question 5 — cas limite : requête hors-sujet

Une recherche sans rapport avec des événements culturels ne doit pas planter — FAISS renverra toujours les k plus proches, même s'ils sont peu pertinents. Utile pour observer le comportement réel plutôt que de le supposer.

In [7]:
run_query("recette de cuisine pour un gâteau au chocolat")

### Question : 'recette de cuisine pour un gâteau au chocolat'

1. Chocolaterie de Puyricard  (score=0.463)
   date  : 2026-06-06T07:30:00+00:00
   lieu  : Aix-en-Provence
   statut: Programmé  | conditions: None
   extrait: 'Chocolaterie de Puyricard\n\nDécouverte du savoir-faire artisanal en chocolaterie avec présentation des techniques de fabrication et dégustation de spéc'

2. Chocolaterie de Puyricard  (score=0.482)
   date  : 2026-06-06T07:00:00+00:00
   lieu  : Aix-en-Provence
   statut: Programmé  | conditions: None
   extrait: 'Chocolaterie de Puyricard\n\nVenez découvrir le savoir-faire traditionnel et artisanal des Maîtres Chocolatiers de notre chocolaterie : techniques du mo'

3. Un projet pour pratiquer  (score=0.489)
   date  : 2026-03-02T16:00:00+00:00
   lieu  : Martigues
   statut: Programmé  | conditions: Gratuit sur inscription
   extrait: "Un projet pour pratiquer\n\nLes Espaces publics numériques de la ville de Martigues accompagnent tous les publics dans l'appropr

## Conclusion

Les 5 questions test confirment que l'index FAISS retrouve des événements globalement pertinents :

- **Q1, Q3** : résultats excellents — thème et lieu correctement croisés (concerts classiques à Marseille, patrimoine/expositions).
- **Q2** : 2 résultats sur 3 très pertinents (jeux gratuits pour enfants) ; le 1er est gratuit mais pas clairement destiné aux enfants — pertinence partielle, pas une erreur.
- **Q4** : les résultats 2 et 3 confirment que le mécanisme mis en place dans `build_text()` fonctionne — un même atelier annulé est retrouvé et correctement étiqueté `"Statut de l'événement : Annulé."`. Le résultat 1 ("Les balades à vélo du MauMa") illustre en revanche une vraie limite : il est remonté à cause du mot "annulation" présent dans son texte, qui décrit en réalité une **politique** d'annulation (info pratique), pas un événement réellement annulé. La recherche sémantique compare des sens de mots, pas des faits vérifiés — démêler cette nuance reviendra au LLM, à l'étape de génération de réponse (API), pas à FAISS seul.
- **Q5** (hors-sujet) : comportement attendu — pas de crash, FAISS renvoie les résultats les plus proches disponibles (chocolaterie, vaguement lié), même sans rapport réel avec la question. C'est la limite documentée de la recherche par similarité seule : elle ne "sait" pas qu'une question sort du domaine couvert par les données.

**Bilan** : la recherche sémantique fonctionne comme attendu sur ce jeu de données. Les limites observées (confusion politique/statut réel, absence de détection du hors-sujet) ne sont pas des bugs à corriger dans FAISS — elles définissent ce que la prochaine étape (génération de réponse via un LLM) devra gérer, en interprétant le contexte récupéré plutôt qu'en s'y fiant aveuglément.
